## Setup e connessione a MongoDB Atlas

In [1]:
import sys
sys.path.append("..")
import os
import numpy as np
import pandas as pd
import sqlite3
from dotenv import load_dotenv
from pymongo import MongoClient

from src.frames import extract_frames
from src.metrics import curve_from_frames, frame_hue_family, color_distribution


In [2]:
# carichiamo la stringa di connessione da .env e ci colleghiamo
load_dotenv("../.env", override=True)
mongo_uri = os.getenv("MONGO_URI")

client = MongoClient(mongo_uri)
db = client["color_in_motion"]
frames_collection = db["frames"]

print("Connessione riuscita")
print("Database esistenti:", client.list_database_names())

Connessione riuscita
Database esistenti: ['color_in_motion', 'mongo_ted', 'sample_mflix', 'admin', 'local']


## Staging: scriviamo ogni fotogramma in MongoDB

Per ogni film, per ogni secondo del trailer, salviamo un documento con le
metriche di colore E il colore RGB pieno (usato per i movie barcode).
Questo e il dato grezzo, non ancora aggregato: lo staging vero del progetto.

In [3]:
# ripuliamo la collezione prima di riscriverla, per evitare duplicati
frames_collection.delete_many({})
print("Collezione pulita. Documenti ora:", frames_collection.count_documents({}))

Collezione pulita. Documenti ora: 0


In [4]:
# lista dei trailer di genere
cartella = "../data/raw/trailers"
files_trailer = []
for f in os.listdir(cartella):
    if f.endswith(".mp4"):
        files_trailer.append(f)

print("Trailer di genere da processare:", len(files_trailer))

Trailer di genere da processare: 133


### Aggiunta: famiglia di tinta e rilevamento frame-cartello

Queste due informazioni sono ora calcolate **per ogni frame** e scritte in
Mongo insieme alle metriche gia' presenti. Motivo: dopo il refactoring del
classificatore (notebook 03), la rappresentazione a "distribuzione per
famiglie di colore" e il filtro dei frame-cartello sono risultati parte
del modello finale scelto (non piu' esperimenti scartati) - quindi, per
la regola gia' scritta in `ARCHITECTURE.md` ("un valore vive nel layer
condiviso solo se serve a piu' di un processo, o e' parte della pipeline
attiva"), meritano di essere promossi da "ricalcolati ad ogni notebook" a
"salvati una volta in Mongo".

`e_uniforme` e `trova_frame_cartello` sono la stessa identica euristica già usata nel notebook 03 (copiata verbatim, non re-implementata), per essere certi che il flag `is_title_card` scritto qui corrisponda esattamente a quello usato per calcolare l'accuratezza del modello finale (34.4% per la catena Distribuzione, dopo la correzione dei bug di allineamento nella cross-validation — vedi 03_color_extraction.ipynb).

In [5]:
def e_uniforme(frame, soglia_frazione=0.55):
    """Un frame e' uniforme se una grande % dei suoi pixel e' vicina al
    colore mediano (sfondo piatto, anche con del testo/logo sopra)."""
    pixel = frame.reshape(-1, 3).astype(float)
    colore_mediano = np.median(pixel, axis=0)
    distanze = np.sqrt(((pixel - colore_mediano) ** 2).sum(axis=1))
    frazione_uniforme = (distanze < 30).mean()
    return frazione_uniforme > soglia_frazione

def trova_frame_cartello(frames):
    """Cammina dall'inizio e dalla fine del trailer, marcando come 'cartello'
    i frame uniformi consecutivi, finche' non trova una scena vera."""
    n = len(frames)
    da_scartare = set()
    i = 0
    while i < n and e_uniforme(frames[i]):
        da_scartare.add(i)
        i += 1
    j = n - 1
    while j >= 0 and j not in da_scartare and e_uniforme(frames[j]):
        da_scartare.add(j)
        j -= 1
    return da_scartare


In [6]:
# scriviamo in mongo un documento per ogni secondo di ogni trailer di genere
# includiamo il colore RGB pieno (mean_colour), la famiglia di tinta
# (hue_family) e se il frame e' un probabile cartello/logo (is_title_card)
fatti = 0
for nome_file in files_trailer:
    tconst = nome_file.replace(".mp4", "")
    percorso = cartella + "/" + nome_file

    frames = extract_frames(percorso, per_second=1)
    brightness, temperature, saturation, colours = curve_from_frames(frames)
    indici_cartello = trova_frame_cartello(frames)

    for secondo in range(len(brightness)):
        colore_rgb = colours[secondo]
        documento = {
            "tconst": tconst,
            "secondo": secondo,
            "brightness": brightness[secondo],
            "temperature": temperature[secondo],
            "saturation": saturation[secondo],
            "mean_colour": [float(colore_rgb[0]), float(colore_rgb[1]), float(colore_rgb[2])],
            "hue_family": frame_hue_family(frames[secondo]),
            "is_title_card": secondo in indici_cartello,
            "gruppo": "genere",
        }
        frames_collection.insert_one(documento)

    fatti = fatti + 1
    if fatti % 20 == 0:
        print("Fatti", fatti, "film...")

print("Finito! Documenti totali nella collezione:", frames_collection.count_documents({}))


Fatti 20 film...
Fatti 40 film...
Fatti 60 film...
Fatti 80 film...
Fatti 100 film...
Fatti 120 film...
Finito! Documenti totali nella collezione: 17642


In [7]:
# stessa cosa per i 9 film d'autore (stesso schema documento, per coerenza -
# non usati dal classificatore, ma manteniamo un'unica forma di documento)
cartella_autore = "../data/raw/auteur_trailers"
files_autore = []
for f in os.listdir(cartella_autore):
    if f.endswith(".mp4"):
        files_autore.append(f)

print("Film d'autore da processare:", len(files_autore))

fatti = 0
for nome_file in files_autore:
    tmdb_id = nome_file.replace(".mp4", "")
    percorso = cartella_autore + "/" + nome_file

    frames = extract_frames(percorso, per_second=1)
    brightness, temperature, saturation, colours = curve_from_frames(frames)
    indici_cartello = trova_frame_cartello(frames)

    for secondo in range(len(brightness)):
        colore_rgb = colours[secondo]
        documento = {
            "tconst": tmdb_id,
            "secondo": secondo,
            "brightness": brightness[secondo],
            "temperature": temperature[secondo],
            "saturation": saturation[secondo],
            "mean_colour": [float(colore_rgb[0]), float(colore_rgb[1]), float(colore_rgb[2])],
            "hue_family": frame_hue_family(frames[secondo]),
            "is_title_card": secondo in indici_cartello,
            "gruppo": "autore",
        }
        frames_collection.insert_one(documento)

    fatti = fatti + 1
    print("Fatto:", tmdb_id)

print("Finito! Documenti totali nella collezione:", frames_collection.count_documents({}))


Film d'autore da processare: 9
Fatto: 110160
Fatto: 1955
Fatto: 660120
Fatto: 38
Fatto: 376386
Fatto: 394117
Fatto: 340485
Fatto: 265177
Fatto: 24469
Finito! Documenti totali nella collezione: 18740


## Aggregazione: MongoDB calcola media e deviazione standard per film

In [8]:
# Pipeline 1: statistiche su TUTTI i frame (come prima), piu' le percentuali
# per famiglia di colore (colore_red, colore_orange, ...) - la stessa
# rappresentazione che nel notebook 03 e' risultata la scelta vincente
# della Fase 1 ("distribuzione dei colori").
pipeline = [
    {
        "$group": {
            "_id": "$tconst",
            "brightness_media": {"$avg": "$brightness"},
            "brightness_std": {"$stdDevPop": "$brightness"},
            "temperature_media": {"$avg": "$temperature"},
            "temperature_std": {"$stdDevPop": "$temperature"},
            "saturation_media": {"$avg": "$saturation"},
            "saturation_std": {"$stdDevPop": "$saturation"},
            "colore_red": {"$avg": {"$cond": [{"$eq": ["$hue_family", "red"]}, 1, 0]}},
            "colore_orange": {"$avg": {"$cond": [{"$eq": ["$hue_family", "orange"]}, 1, 0]}},
            "colore_yellow": {"$avg": {"$cond": [{"$eq": ["$hue_family", "yellow"]}, 1, 0]}},
            "colore_green": {"$avg": {"$cond": [{"$eq": ["$hue_family", "green"]}, 1, 0]}},
            "colore_blue": {"$avg": {"$cond": [{"$eq": ["$hue_family", "blue"]}, 1, 0]}},
            "colore_purple": {"$avg": {"$cond": [{"$eq": ["$hue_family", "purple"]}, 1, 0]}},
            "colore_dark_neutral": {"$avg": {"$cond": [{"$eq": ["$hue_family", "dark_neutral"]}, 1, 0]}},
            "pct_title_card": {"$avg": {"$cond": ["$is_title_card", 1, 0]}},
            "n_secondi": {"$sum": 1},
        }
    }
]

risultati_mongo = list(frames_collection.aggregate(pipeline))
print("Film aggregati:", len(risultati_mongo))
print(risultati_mongo[0])


Film aggregati: 142
{'_id': 'tt1386697', 'brightness_media': 11.061550120950386, 'brightness_std': 9.442641553076918, 'temperature_media': 2.3113522904672363, 'temperature_std': 3.436316254059827, 'saturation_media': 0.238918508341992, 'saturation_std': 0.10435307448382847, 'colore_red': 0.0, 'colore_orange': 0.02403846153846154, 'colore_yellow': 0.0625, 'colore_green': 0.04807692307692308, 'colore_blue': 0.0, 'colore_purple': 0.0, 'colore_dark_neutral': 0.8653846153846154, 'pct_title_card': 0.1201923076923077, 'n_secondi': 208}


### Pipeline 2: le stesse statistiche, escludendo i frame-cartello

Corrisponde allo step "+ Filtro cartelli" del notebook 03, che nella
catena finale ha fatto salire l'accuratezza. Un `$match` sui documenti
con `is_title_card: false` prima del `$group` equivale esattamente a
"ricalcolare la distribuzione colori sui frame puliti", ma qui lo fa
MongoDB una volta sola invece che il notebook ad ogni esecuzione.

In [9]:
pipeline_filtrata = [
    {"$match": {"is_title_card": False}},
    {
        "$group": {
            "_id": "$tconst",
            "brightness_media_filtrato": {"$avg": "$brightness"},
            "brightness_std_filtrato": {"$stdDevPop": "$brightness"},
            "temperature_media_filtrato": {"$avg": "$temperature"},
            "temperature_std_filtrato": {"$stdDevPop": "$temperature"},
            "saturation_media_filtrato": {"$avg": "$saturation"},
            "saturation_std_filtrato": {"$stdDevPop": "$saturation"},
            "colore_red_filtrato": {"$avg": {"$cond": [{"$eq": ["$hue_family", "red"]}, 1, 0]}},
            "colore_orange_filtrato": {"$avg": {"$cond": [{"$eq": ["$hue_family", "orange"]}, 1, 0]}},
            "colore_yellow_filtrato": {"$avg": {"$cond": [{"$eq": ["$hue_family", "yellow"]}, 1, 0]}},
            "colore_green_filtrato": {"$avg": {"$cond": [{"$eq": ["$hue_family", "green"]}, 1, 0]}},
            "colore_blue_filtrato": {"$avg": {"$cond": [{"$eq": ["$hue_family", "blue"]}, 1, 0]}},
            "colore_purple_filtrato": {"$avg": {"$cond": [{"$eq": ["$hue_family", "purple"]}, 1, 0]}},
            "colore_dark_neutral_filtrato": {"$avg": {"$cond": [{"$eq": ["$hue_family", "dark_neutral"]}, 1, 0]}},
            "n_secondi_filtrato": {"$sum": 1},
        }
    }
]

risultati_mongo_filtrati = list(frames_collection.aggregate(pipeline_filtrata))
print("Film aggregati (senza frame-cartello):", len(risultati_mongo_filtrati))
print(risultati_mongo_filtrati[0])


Film aggregati (senza frame-cartello): 142
{'_id': 'tt5140878', 'brightness_media_filtrato': 9.585367480626237, 'brightness_std_filtrato': 9.268516846631659, 'temperature_media_filtrato': 3.402786540566193, 'temperature_std_filtrato': 3.35660357015158, 'saturation_media_filtrato': 0.3375173830306737, 'saturation_std_filtrato': 0.12889263058000072, 'colore_red_filtrato': 0.0, 'colore_orange_filtrato': 0.09166666666666666, 'colore_yellow_filtrato': 0.075, 'colore_green_filtrato': 0.016666666666666666, 'colore_blue_filtrato': 0.0, 'colore_purple_filtrato': 0.0, 'colore_dark_neutral_filtrato': 0.8166666666666667, 'n_secondi_filtrato': 120}


## Warehouse: scriviamo il risultato aggregato in SQLite

In [10]:
# uniamo le medie di Mongo (pipeline 1 + pipeline 2 filtrata) con
# titolo/genere/anno (dal Source)
mongo_df = pd.DataFrame(risultati_mongo)
mongo_df = mongo_df.rename(columns={"_id": "tconst"})

mongo_df_filtrato = pd.DataFrame(risultati_mongo_filtrati)
mongo_df_filtrato = mongo_df_filtrato.rename(columns={"_id": "tconst"})

mongo_completo = mongo_df.merge(mongo_df_filtrato, on="tconst", how="left")

info_film = pd.read_csv("../data/source/film_with_trailers.csv")
info_film = info_film[["tconst", "primaryTitle", "genere_principale", "startYear"]]

info_autore = pd.read_csv("../data/source/auteur_films.csv")
info_autore = info_autore.rename(columns={"titolo": "primaryTitle", "anno": "startYear", "tmdb_id": "tconst"})
info_autore["tconst"] = info_autore["tconst"].astype(str)
info_autore["genere_principale"] = "Auteur"
info_autore = info_autore[["tconst", "primaryTitle", "genere_principale", "startYear"]]

tutte_info = pd.concat([info_film, info_autore], ignore_index=True)
tabella_finale = mongo_completo.merge(tutte_info, on="tconst", how="left")

print("Righe finali:", len(tabella_finale))
print("Colonne totali:", len(tabella_finale.columns))
tabella_finale.head()


Righe finali: 142
Colonne totali: 33


,tconst,brightness_media,brightness_std,temperature_media,temperature_std,saturation_media,saturation_std,colore_red,colore_orange,colore_yellow,...,colore_orange_filtrato,colore_yellow_filtrato,colore_green_filtrato,colore_blue_filtrato,colore_purple_filtrato,colore_dark_neutral_filtrato,n_secondi_filtrato,primaryTitle,genere_principale,startYear
0,tt1386697,11.061550,9.442642,2.311352,3.436316,0.238919,0.104353,0.0,0.024038,0.062500,...,0.027322,0.071038,0.054645,0.000000,0.0,0.846995,183,Suicide Squad,Action,2016
1,tt0266697,35.880742,19.002602,13.127373,14.693992,0.444146,0.183815,0.0,0.077922,0.383117,...,0.085714,0.407143,0.357143,0.042857,0.0,0.107143,140,Kill Bill: Vol. 1,Action,2003
2,tt0121766,17.152618,14.418586,2.697339,10.230690,0.232491,0.157307,0.0,0.044872,0.012821,...,0.050725,0.014493,0.101449,0.036232,0.0,0.797101,138,Star Wars: Episode III - Revenge of the Sith,Action,2005
3,tt0097165,29.529438,22.901204,7.283140,11.719100,0.384079,0.143985,0.0,0.005556,0.205556,...,0.000000,0.118056,0.583333,0.090278,0.0,0.208333,144,Dead Poets Society,Comedy,1989
4,tt0375679,18.845243,14.084286,3.997681,4.674487,0.332588,0.099343,0.0,0.162162,0.148649,...,0.206897,0.189655,0.163793,0.017241,0.0,0.422414,116,Crash,Crime,2004


In [11]:
conn = sqlite3.connect("../data/warehouse/color_in_motion.db")
tabella_finale.to_sql("films", conn, if_exists="replace", index=False)

print("Tabella 'films' in SQLite aggiornata da MongoDB")
print("Righe:", len(tabella_finale))

check = pd.read_sql_query("SELECT COUNT(*) as n FROM films", conn)
print(check)

Tabella 'films' in SQLite aggiornata da MongoDB
Righe: 142
     n
0  142


## Validazione: le medie di MongoDB coincidono con quelle calcolate in Python?

In [12]:
# Confronto vero (non solo un elenco di colonne): per 3 film campione,
# ricalcoliamo in Python, dal video sorgente, sia la distribuzione colori
# sia le statistiche filtrate (senza frame-cartello) - stesso principio
# gia' usato per brightness/temperature/saturation - e confrontiamo con
# cio' che e' finito nel warehouse attraverso l'aggregazione Mongo.
campione = tabella_finale[tabella_finale["genere_principale"] != "Auteur"].sample(3, random_state=42)

for _, riga in campione.iterrows():
    tconst = riga["tconst"]
    nome_file = tconst + ".mp4"
    frames = extract_frames(cartella + "/" + nome_file, per_second=1)
    indici_cartello = trova_frame_cartello(frames)
    frames_puliti = [f for i, f in enumerate(frames) if i not in indici_cartello]
    if len(frames_puliti) == 0:
        frames_puliti = frames

    dist_python = color_distribution(frames)
    dist_python_filtrato = color_distribution(frames_puliti)
    _, temp_python_filtrato, _, _ = curve_from_frames(frames_puliti)

    print(f"--- {tconst} ---")
    print("  colore_red             - Mongo:", round(riga["colore_red"], 4),
          " | Python:", round(dist_python["red"], 4))
    print("  colore_red (filtrato)  - Mongo:", round(riga["colore_red_filtrato"], 4),
          " | Python:", round(dist_python_filtrato["red"], 4))
    print("  temperature_media (filtrato) - Mongo:", round(riga["temperature_media_filtrato"], 2),
          " | Python:", round(np.mean(temp_python_filtrato), 2))
    print()

print("Se i valori Mongo e Python coincidono (a parte arrotondamenti), la")
print("pipeline hue_family/is_title_card -> aggregazione e' corretta end-to-end.")


--- tt1270798 ---
  colore_red             - Mongo: 0.0  | Python: 0.0
  colore_red (filtrato)  - Mongo: 0.0  | Python: 0.0
  temperature_media (filtrato) - Mongo: 4.67  | Python: 4.67

--- tt1706593 ---
  colore_red             - Mongo: 0.0  | Python: 0.0
  colore_red (filtrato)  - Mongo: 0.0  | Python: 0.0
  temperature_media (filtrato) - Mongo: 1.4  | Python: 1.4

--- tt3289956 ---
  colore_red             - Mongo: 0.0  | Python: 0.0
  colore_red (filtrato)  - Mongo: 0.0  | Python: 0.0
  temperature_media (filtrato) - Mongo: 0.08  | Python: 0.08

Se i valori Mongo e Python coincidono (a parte arrotondamenti), la
pipeline hue_family/is_title_card -> aggregazione e' corretta end-to-end.


## Query SQL sul warehouse (esempio RDBMS)

In [13]:
query = """
SELECT genere_principale,
       ROUND(AVG(brightness_media), 1) AS luminosita_media,
       COUNT(*) AS n_film
FROM films
WHERE genere_principale != 'Auteur'
GROUP BY genere_principale
ORDER BY luminosita_media
"""

risultato = pd.read_sql_query(query, conn)
print(risultato)

  genere_principale  luminosita_media  n_film
0            Horror              12.3      18
1             Crime              18.9      19
2            Action              19.4      72
3             Drama              20.4      11
4            Comedy              23.6      13


### Seconda query: le nuove feature sono interrogabili in SQL

A dimostrazione che la promozione a colonne del warehouse funziona: la
percentuale di frame rossi per genere, e quanto in media un trailer e'
fatto di frame-cartello (`pct_title_card`) - prima questi valori
esistevano solo dentro un notebook, ora sono una query SQL.

In [14]:
query_colore = """
SELECT genere_principale,
       ROUND(AVG(colore_red) * 100, 1) AS pct_rosso_medio,
       ROUND(AVG(colore_dark_neutral) * 100, 1) AS pct_scuro_medio,
       ROUND(AVG(pct_title_card) * 100, 1) AS pct_frame_cartello_medio
FROM films
WHERE genere_principale != 'Auteur'
GROUP BY genere_principale
ORDER BY pct_rosso_medio DESC
"""

risultato_colore = pd.read_sql_query(query_colore, conn)
print(risultato_colore)


  genere_principale  pct_rosso_medio  pct_scuro_medio  \
0             Crime              0.3             57.9   
1            Action              0.3             58.0   
2            Horror              0.1             77.7   
3             Drama              0.1             53.9   
4            Comedy              0.0             42.9   

   pct_frame_cartello_medio  
0                      12.7  
1                      11.7  
2                      22.8  
3                      17.6  
4                       9.8  
